# 1. Material Lookup Tool (ASME Section II Part D Database)

In [2]:
# tools/material_lookup_tool.py

MATERIAL_DB = {
    "2.25Cr-1Mo": {
        "material": "2.25Cr-1Mo (SA-387 Gr 22)",
        "temperatures": {
            350.0: {
                "allowable_stress_mpa": 138.0,
                "yield_strength_mpa": 310.0,
                "tensile_strength_mpa": 515.0
            }
        }
    },

    "SA-516 Gr 70": {
        "material": "SA-516 Gr 70",
        "temperatures": {
            350.0: {
                "allowable_stress_mpa": 120.0,
                "yield_strength_mpa": 260.0,
                "tensile_strength_mpa": 485.0
            }
        }
    },

    "SA-240 316L": {
        "material": "SA-240 316L",
        "temperatures": {
            350.0: {
                "allowable_stress_mpa": 100.0,
                "yield_strength_mpa": 170.0,
                "tensile_strength_mpa": 485.0
            }
        }
    }
}


def lookup_material(
    material_grade: str,
    temperature_c: float
) -> dict:
    if not isinstance(material_grade, str):
        return {
            "status": "ERROR",
            "error": "Material grade must be a string."
        }

    material_grade = material_grade.strip()

    if not material_grade:
        return {
            "status": "ERROR",
            "error": "Material grade cannot be empty."
        }

    if not isinstance(temperature_c, (int, float)):
        return {
            "status": "ERROR",
            "error": "Temperature must be a number."
        }

    if temperature_c != temperature_c:
        return {
            "status": "ERROR",
            "error": "Temperature cannot be NaN."
        }

    if material_grade not in MATERIAL_DB:
        return {
            "status": "ERROR",
            "error": f"Material '{material_grade}' not found in offline database."
        }

    material = MATERIAL_DB[material_grade]

    if temperature_c not in material["temperatures"]:
        return {
            "status": "ERROR",
            "error": (
                f"No verified data available for "
                f"{material_grade} at {temperature_c} °C."
            )
        }

    data = material["temperatures"][temperature_c]

    if data["allowable_stress_mpa"] <= 0:
        return {
            "status": "ERROR",
            "error": "Invalid allowable stress in database."
        }

    return {
        "status": "SUCCESS",
        "material": material["material"],
        "temperature_c": temperature_c,
        "allowable_stress_mpa": data["allowable_stress_mpa"],
        "yield_strength_mpa": data["yield_strength_mpa"],
        "tensile_strength_mpa": data["tensile_strength_mpa"]
    }


def lookup_allowable_stress(
    material_grade: str,
    temperature_c: float
) -> float:
    result = lookup_material(material_grade, temperature_c)
    if result["status"] == "ERROR":
        raise ValueError(result["error"])
    return result["allowable_stress_mpa"]

# Test Example
stress = lookup_allowable_stress("2.25Cr-1Mo", temperature_c=350.0)
print(f"Allowable Stress: {stress} MPa")


Allowable Stress: 138.0 MPa


# 2. API 579 Fitness-For-Service (FFS) Tool

In [3]:
# tools/api_579_ffs_tool.py

import math

ALLOWABLE_RSF = 0.90

def evaluate_lta(
    t_actual_global: float,
    t_min_lta: float,
    t_req: float,
    flaw_length_mm: float,
    inside_radius_mm: float
) -> dict:
    """
    API 579-1 / ASME FFS-1 Level 1 style
    Localized Thin Area (LTA) assessment.
    """
    values = {
        "t_actual_global": t_actual_global,
        "t_min_lta": t_min_lta,
        "t_req": t_req,
        "flaw_length_mm": flaw_length_mm,
        "inside_radius_mm": inside_radius_mm
    }

    for name, value in values.items():
        if isinstance(value, bool):
            return {
                "status": "ERROR",
                "error": f"{name} must be a numeric value, not boolean."
            }
        if not isinstance(value, (int, float)):
            return {
                "status": "ERROR",
                "error": f"{name} must be a numeric value."
            }
        if not math.isfinite(value):
            return {
                "status": "ERROR",
                "error": f"{name} cannot be NaN or infinity."
            }

    if t_actual_global <= 0:
        return {"status": "ERROR", "error": "Global thickness must be greater than 0 mm."}
    if t_min_lta <= 0:
        return {"status": "ERROR", "error": "Minimum LTA thickness must be greater than 0 mm."}
    if t_req <= 0:
        return {"status": "ERROR", "error": "Required thickness must be greater than 0 mm."}
    if flaw_length_mm <= 0:
        return {"status": "ERROR", "error": "Flaw length must be greater than 0 mm."}
    if inside_radius_mm <= 0:
        return {"status": "ERROR", "error": "Inside radius must be greater than 0 mm."}
    if t_min_lta > t_actual_global:
        return {"status": "ERROR", "error": "Minimum LTA thickness cannot be greater than global thickness."}

    Rt = t_min_lta / t_req
    if Rt <= 0:
        return {"status": "ERROR", "error": "Remaining thickness ratio Rt must be greater than 0."}
    if Rt < 0.20:
        return {
            "status": "ERROR",
            "error": f"Rt = {Rt:.3f} is below Level 1 screening limits (0.20). Level 2/3 required."
        }

    diameter_mm = 2.0 * inside_radius_mm
    sqrt_argument = diameter_mm * t_req
    lambda_value = 1.285 * flaw_length_mm / math.sqrt(sqrt_argument)
    Mt = math.sqrt(1.0 + (0.48 * lambda_value ** 2))
    denominator = 1.0 - ((1.0 - Rt) / Mt)
    if denominator <= 0:
        return {"status": "ERROR", "error": "Invalid calculation parameters for RSF denominator."}

    rsf = Rt / denominator
    is_acceptable = rsf >= ALLOWABLE_RSF

    if is_acceptable:
        action = "Acceptable for continued operation based on Level 1 RSF assessment."
    else:
        action = "Pressure derating to 118 barg or weld overlay required before turnaround."

    return {
        "rsf": round(rsf, 3),
        "allowable_rsf": ALLOWABLE_RSF,
        "is_acceptable": is_acceptable,
        "action": action
    }

# Test Example
verdict = evaluate_lta(140.0, 138.2, 138.57, 50.0, 1200.0)
print("FFS Verdict:", verdict)


FFS Verdict: {'rsf': 1.0, 'allowable_rsf': 0.9, 'is_acceptable': True, 'action': 'Acceptable for continued operation based on Level 1 RSF assessment.'}


# 3. API 581 Risk-Based Inspection (RBI) Tool

In [4]:
# tools/risk_based_inspection_tool.py

import math

def calculate_rbi_interval(
    remaining_life_years: float,
    design_pressure_mpa: float,
    fluid_toxicity: str
) -> dict:
    """
    RBI screening / interval engine.
    """
    if isinstance(remaining_life_years, bool):
        return {"status": "ERROR", "error": "remaining_life_years must be a number."}
    if not isinstance(remaining_life_years, (int, float)):
        return {"status": "ERROR", "error": "remaining_life_years must be a number."}
    if isinstance(design_pressure_mpa, bool):
        return {"status": "ERROR", "error": "design_pressure_mpa must be a number."}
    if not isinstance(design_pressure_mpa, (int, float)):
        return {"status": "ERROR", "error": "design_pressure_mpa must be a number."}
    if not isinstance(fluid_toxicity, str):
        return {"status": "ERROR", "error": "fluid_toxicity must be a string."}

    if not math.isfinite(remaining_life_years):
        return {"status": "ERROR", "error": "remaining_life_years cannot be NaN or infinity."}
    if not math.isfinite(design_pressure_mpa):
        return {"status": "ERROR", "error": "design_pressure_mpa cannot be NaN or infinity."}
    if design_pressure_mpa <= 0:
        return {"status": "ERROR", "error": "Design pressure must be greater than 0 MPa."}

    fluid_toxicity = fluid_toxicity.strip().lower()
    if not fluid_toxicity:
        return {"status": "ERROR", "error": "Fluid toxicity cannot be empty."}
    allowed_toxicity = {"low", "medium", "high"}
    if fluid_toxicity not in allowed_toxicity:
        return {"status": "ERROR", "error": "fluid_toxicity must be 'Low', 'Medium', or 'High'."}

    if remaining_life_years <= 0:
        return {
            "risk_category": "CRITICAL_HIGH",
            "recommended_interval_months": 0,
            "statutory_verdict": "MANDATORY_IMMEDIATE_SHUTDOWN_INSPECTION"
        }

    if fluid_toxicity == "high": toxicity_factor = 3
    elif fluid_toxicity == "medium": toxicity_factor = 2
    else: toxicity_factor = 1

    if design_pressure_mpa >= 15: pressure_factor = 3
    elif design_pressure_mpa >= 5: pressure_factor = 2
    else: pressure_factor = 1

    if remaining_life_years <= 1: life_factor = 3
    elif remaining_life_years <= 3: life_factor = 2
    else: life_factor = 1

    risk_score = toxicity_factor + pressure_factor + life_factor

    if risk_score >= 8:
        risk_category = "HIGH"
        recommended_interval_months = 6
        statutory_verdict = "ENGINEERING_REVIEW_REQUIRED"
    elif risk_score >= 6:
        risk_category = "MEDIUM_HIGH"
        recommended_interval_months = 12
        statutory_verdict = "ENHANCED_INSPECTION_REQUIRED"
    elif risk_score >= 4:
        risk_category = "MEDIUM"
        recommended_interval_months = 24
        statutory_verdict = "PERIODIC_INSPECTION_REQUIRED"
    else:
        risk_category = "LOW"
        recommended_interval_months = 36
        statutory_verdict = "ROUTINE_INSPECTION"

    return {
        "risk_category": risk_category,
        "recommended_interval_months": recommended_interval_months,
        "statutory_verdict": statutory_verdict
    }

# Test Example from README
interval = calculate_rbi_interval(
    remaining_life_years=-0.49,
    design_pressure_mpa=14.5,
    fluid_toxicity="High"
)
print("RBI Interval Output:", interval)


RBI Interval Output: {'risk_category': 'CRITICAL_HIGH', 'recommended_interval_months': 0, 'statutory_verdict': 'MANDATORY_IMMEDIATE_SHUTDOWN_INSPECTION'}
